# Notebook Cost Demo
This notebook generates warehouse pushdown spend for cost attribution testing.

In [ ]:
-- Generate some warehouse compute by querying large system views
SELECT 
    query_id,
    query_text,
    database_name,
    schema_name,
    user_name,
    warehouse_name,
    execution_time,
    total_elapsed_time
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
ORDER BY total_elapsed_time DESC
LIMIT 1000;

In [ ]:
-- Cross-join to generate compute-intensive workload
WITH numbers AS (
    SELECT SEQ4() AS n FROM TABLE(GENERATOR(ROWCOUNT => 10000))
),
aggregates AS (
    SELECT 
        n,
        HASH(n) AS hash_val,
        SQRT(ABS(HASH(n))) AS sqrt_val,
        LN(ABS(HASH(n)) + 1) AS log_val,
        MOD(HASH(n), 100) AS mod_val
    FROM numbers
)
SELECT 
    MOD(n, 50) AS bucket,
    COUNT(*) AS cnt,
    AVG(sqrt_val) AS avg_sqrt,
    STDDEV(log_val) AS std_log,
    MEDIAN(mod_val) AS median_mod,
    APPROX_PERCENTILE(hash_val, 0.95) AS p95_hash
FROM aggregates
GROUP BY bucket
ORDER BY bucket;

In [ ]:
-- Another compute-intensive query scanning account_usage
SELECT 
    DATE_TRUNC('hour', start_time) AS hour,
    warehouse_name,
    COUNT(*) AS query_count,
    AVG(execution_time) AS avg_execution_ms,
    SUM(bytes_scanned) AS total_bytes_scanned,
    SUM(rows_produced) AS total_rows_produced
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD(day, -30, CURRENT_TIMESTAMP())
GROUP BY 1, 2
HAVING COUNT(*) > 5
ORDER BY total_bytes_scanned DESC
LIMIT 500;

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()

df = session.sql("""
    SELECT 
        user_name,
        COUNT(*) as query_count,
        SUM(credits_used_cloud_services) as cloud_credits
    FROM snowflake.account_usage.query_history
    WHERE start_time >= DATEADD(day, -30, CURRENT_TIMESTAMP())
    GROUP BY user_name
    ORDER BY query_count DESC
    LIMIT 100
""")

result = df.to_pandas()
print(f"Found {len(result)} users with query activity")
result.head(10)

In [ ]:
-- Large scan of warehouse metering history
SELECT 
    warehouse_name,
    DATE_TRUNC('day', start_time) AS day,
    SUM(credits_used) AS daily_credits,
    SUM(credits_used_compute) AS compute_credits,
    SUM(credits_used_cloud_services) AS cloud_credits
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD(day, -90, CURRENT_TIMESTAMP())
GROUP BY 1, 2
ORDER BY daily_credits DESC;

In [ ]:
login_df = session.sql("""
    SELECT 
        user_name,
        client_ip,
        reported_client_type,
        first_authentication_factor,
        event_timestamp
    FROM snowflake.account_usage.login_history
    WHERE event_timestamp >= DATEADD(day, -30, CURRENT_TIMESTAMP())
    ORDER BY event_timestamp DESC
    LIMIT 5000
""")

login_result = login_df.to_pandas()
print(f"Processed {len(login_result)} login events")